In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [6]:
def train_torch_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=50):
    model.to(device)
    history = {'train_loss':[], 'val_loss':[]}
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()*xb.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        # validation loss
        model.eval()
        with torch.no_grad():
            val_running = 0.0
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_running += loss.item()*xb.size(0)
            val_loss = val_running / len(val_loader.dataset)
            history['val_loss'].append(val_loss)
    return history

In [7]:
def partB_breast_cancer(device=torch.device('cpu')):
    data = load_breast_cancer()
    X = data.data
    y = data.target.reshape(-1,1).astype(np.float32)  
    
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.2, random_state=42, stratify=y)
    
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.float32)

    train_ds = TensorDataset(Xtr_t, ytr_t)
    test_ds = TensorDataset(Xte_t, yte_t)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64)

    
    class BreastNet(nn.Module):
        def __init__(self, in_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
                nn.Sigmoid()
            )
        def forward(self, x):
            return self.net(x)

    model = BreastNet(X.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    history = train_torch_model(model, train_loader, test_loader, criterion, optimizer, device, epochs=50)

    model.eval()
    with torch.no_grad():
        preds_tr = (model(Xtr_t.to(device)).cpu().numpy() >= 0.5).astype(int)
        preds_te = (model(Xte_t.to(device)).cpu().numpy() >= 0.5).astype(int)
    acc_tr = (preds_tr.flatten() == ytr.flatten()).mean()
    acc_te = (preds_te.flatten() == yte.flatten()).mean()
    print('\nBreast Cancer - Train acc: {:.4f}, Test acc: {:.4f}'.format(acc_tr, acc_te))
    return model, history

In [8]:
# If you don't have GPU change device to torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if __name__ == '__main__':
    device = torch.device('cuda')
    print('Using device:', device)
    model_bc, hist_bc = partB_breast_cancer(device)

Using device: cuda


AssertionError: Torch not compiled with CUDA enabled